In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Users\hp\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\hp\anaconda3\envs\pyspark_env\python.exe"
print(" Environment Ready!")


 Environment Ready!


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01 - Data Ingestion") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(" Spark Ready! Version:", spark.version)

 Spark Ready! Version: 4.1.1


In [5]:
from pyspark.sql.functions import col, lower, regexp_replace, trim

df = spark.read.csv(
    r"C:\Users\hp\BIGDATAPROJECT\data\Tweets.csv",
    header=True,
    inferSchema=True
)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()
df.show(5)

Total rows: 14837
Total columns: 15
root
 |-- tweet_id: string (nullable = true)
 |-- airline_sentiment: string (nullable = true)
 |-- airline_sentiment_confidence: string (nullable = true)
 |-- negativereason: string (nullable = true)
 |-- negativereason_confidence: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- airline_sentiment_gold: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negativereason_gold: string (nullable = true)
 |-- retweet_count: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- tweet_coord: string (nullable = true)
 |-- tweet_created: string (nullable = true)
 |-- tweet_location: string (nullable = true)
 |-- user_timezone: string (nullable = true)

+------------------+-----------------+----------------------------+--------------+-------------------------+--------------+----------------------+----------+-------------------+-------------+--------------------+-----------+--------------------+--------------+-----

In [6]:
# Select only needed columns
df_clean = df.select(
    "tweet_id", "airline_sentiment", "airline_sentiment_confidence",
    "negativereason", "airline", "retweet_count",
    "text", "tweet_created", "tweet_location"
)

# Drop nulls in key columns
df_clean = df_clean.dropna(subset=["tweet_id", "airline_sentiment", "airline", "text"])

# Clean text — remove URLs, @mentions, special characters
df_clean = df_clean.withColumn("clean_text",
    trim(lower(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("text"), r"http\S+", ""),
            r"@\w+", ""),
        r"[^a-zA-Z\s]", "")
    ))
)

print(" Cleaned rows:", df_clean.count())
df_clean.show(5, truncate=60)

 Cleaned rows: 14632
+------------------+-----------------+----------------------------+--------------+--------------+-------------+------------------------------------------------------------+-------------------------+--------------+------------------------------------------------------------+
|          tweet_id|airline_sentiment|airline_sentiment_confidence|negativereason|       airline|retweet_count|                                                        text|            tweet_created|tweet_location|                                                  clean_text|
+------------------+-----------------+----------------------------+--------------+--------------+-------------+------------------------------------------------------------+-------------------------+--------------+------------------------------------------------------------+
|570306133677760513|          neutral|                         1.0|          NULL|Virgin America|            0|                         @VirginAmerica Wha

In [7]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["tweets_db"]
collection = db["airline_tweets"]

# Drop old collection if re-running
collection.drop()

# Insert data
pandas_df = df_clean.toPandas()
records = pandas_df.to_dict("records")
collection.insert_many(records)

print(f" Inserted {collection.count_documents({})} records into MongoDB!")


 Inserted 14632 records into MongoDB!


In [8]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["tweets_db"]
collection = db["airline_tweets"]

# Drop old collection
collection.drop()

# Convert to pandas
pandas_df = df_clean.toPandas()

#  Remove duplicate tweet_ids before inserting
pandas_df = pandas_df.drop_duplicates(subset=["tweet_id"])
pandas_df = pandas_df.dropna(subset=["tweet_id"])

# Insert data
records = pandas_df.to_dict("records")
collection.insert_many(records)

print(f" Inserted {collection.count_documents({})} records into MongoDB!")
print(f"Total records after dedup: {len(records)}")

 Inserted 14478 records into MongoDB!
Total records after dedup: 14478


In [12]:
# Basic indexes — no unique on tweet_id to avoid duplicate issues
collection.create_index("airline")
collection.create_index("airline_sentiment")
collection.create_index("negativereason")
collection.create_index([("airline", 1), ("airline_sentiment", 1)])

# tweet_id index — safe version
collection.create_index("tweet_id", unique=False)  # ← changed to False

print(" Indexes created!")
print("Index list:", list(collection.index_information().keys()))



 Indexes created!
Index list: ['_id_', 'airline_1', 'airline_sentiment_1', 'negativereason_1', 'airline_1_airline_sentiment_1', 'tweet_id_1']


In [13]:
# Check duplicates in original data
print("Total rows:", len(pandas_df))
print("Unique tweet_ids:", pandas_df["tweet_id"].nunique())
print("Duplicates found:", len(pandas_df) - pandas_df["tweet_id"].nunique())

Total rows: 14478
Unique tweet_ids: 14478
Duplicates found: 0
